In [1]:
# !pip install pyspark

import findspark

# fetch from the local install...
loc_sparkpath = '/Users/pneuma/bigdatapro/spark-3.5.1-bin-hadoop3'
findspark.init(loc_sparkpath) #('/home/matthew/spark-2.1.0-bin-hadoop2.7')


In [2]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import explode, split, lower, col, desc


In [ ]:

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("HashtagCount") \
    .getOrCreate()

# Read Streaming data from a TCP source
lines = spark.readStream \
    .format("socket") \
    .option("host", "127.0.0.1") \
    .option("port", 5554) \
    .load()

# Split lines into words, filter hashtags, count each hashtag
hashtags = lines.select(
   explode(
       split(lines.value, " ")
   ).alias("word")
).filter(
   lower(col("word")).startswith("#")
).groupBy(
   lower(col("word")).alias("hashtag")
).count().alias("count")

# Sort hashtags by count in descending order
sortedHashtags = hashtags.orderBy(desc("count"))

# Start query stream that writes to console and await termination
query = sortedHashtags.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/08 14:55:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/04/08 14:55:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/04/08 14:55:34 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/04/08 14:55:34 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
24/04/08 14:55:35 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/45/fyyfd4dj2s16s3sz5vftgt600000gn/T/temporary-058eca81-8eb1-47ff-a95f-918b6b222f07. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to 

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|hashtag|count|
+-------+-----+
+-------+-----+



In [ ]:
# start the listener now --- Now run TweetListener.py

In [ ]:
spark.start()

In [ ]:
import time
from IPython import display
import matplotlib.pyplot as plt
import seaborn as sns
# Only works for Jupyter Notebooks!
%matplotlib inline 

In [ ]:
count = 0
while count < 10:
    
    time.sleep( 3 )
    top_10_tweets = sqlContext.sql( 'Select tag, count from tweets' )
    top_10_df = top_10_tweets.toPandas()
    display.clear_output(wait=True)
    plt.figure( figsize = ( 10, 8 ) )
#     sns.barplot(x='count',y='land_cover_specific', data=df, palette='Spectral')
    sns.barplot( x="count", y="tag", data=top_10_df)
    plt.show()
    count = count + 1